# T06 — Paleobathymetry profile across the Atlantic at 50 Ma

**Half-space cooling, applied to a reconstructed age grid, is paleo-*basement* depth — not true paleobathymetry.** True paleobathymetry would require adding the sediment column on top of the basement; the GDH cooling model alone gives only the top of the oceanic crust as a function of age.

## What this notebook produces

A single chart panel showing a **long bathymetric profile** from the paleo-position of New York to the paleo-position of Lisbon at 50 Ma. The profile is extracted from a paleo-depth grid built by feeding the Zahirovic 2022 reconstructed age grid through the GDH half-space cooling model.

**Audience**: postgraduate / honours.  
**Difficulty**: ★★★.

In [ ]:
# Cell 1 — imports + reconstruct endpoints
import numpy as np, pandas as pd, xarray as xr, gplately, pygmt
from plate_model_manager import PlateModelManager
from IPython.display import display, HTML

T = 50
pmm = PlateModelManager()
model = pmm.get_model("Zahirovic2022", data_dir="./gplately_data")
recon = gplately.PlateReconstruction(
    rotation_model=model.get_rotation_model(),
    topology_features=model.get_topologies(),
    static_polygons=model.get_static_polygons(),
)

# Present-day NYC & Lisbon
ENDS = gplately.Points(plate_reconstruction=recon,
                        lons=[-74.0, -9.1], lats=[40.7, 38.7],
                        plate_id=[101, 301])
end_lats, end_lons = ENDS.reconstruct(time=T, return_array=True)
print(f"NYC paleo @ {T} Ma:    ({end_lats[0]:.1f}, {end_lons[0]:.1f})")
print(f"Lisbon paleo @ {T} Ma: ({end_lats[1]:.1f}, {end_lons[1]:.1f})")

In [ ]:
# Cell 2 — load age grid → paleo-basement depth (sediment-free)
age_file = model.get_raster("AgeGrids", time=T)
age_da = xr.open_dataarray(age_file) if str(age_file).endswith(".nc") \
         else xr.open_dataset(age_file).to_array().isel(variable=0)

def age_to_depth(a): return -(2500 + 350 * np.sqrt(np.clip(a, 0, None)))
depth_da = xr.apply_ufunc(age_to_depth, age_da)

### What the profile shows

Read the profile left-to-right. The two ends sit on the **continental margins** (shallow, near the paleo-shorelines of North America and Iberia). Walking inboard from either margin you cross the **continental shelf** edge and drop into deep ocean. The deep ocean reaches a maximum depth at ≈4 km below sea level. This is the old, cold crust that was created during early Atlantic opening (≈150 Ma).

The bathymetric **high in the middle** of the profile is the **paleo-mid-Atlantic ridge**: young, hot, buoyant crust sitting ≈2.5 km below sea level. The symmetry of the depth field about the ridge is the visual signature of symmetrical seafloor spreading.

In [ ]:
# Cell 3 — profile via pygmt.project + grdtrack
#
# `pygmt.grdtrack` only honours `newcolname` when `points` is a DataFrame
# with named columns. Passing a bare numpy.column_stack falls back to
# integer column labels (0, 1, 2), so `profile_track["depth"]` would
# raise KeyError. Build a named DataFrame to avoid that.
pts = pd.DataFrame({
    "lon": np.linspace(end_lons[0], end_lons[1], 400),
    "lat": np.linspace(end_lats[0], end_lats[1], 400),
})
profile_track = pygmt.grdtrack(
    points=pts, grid=depth_da, newcolname="depth",
)

# Drop samples where grdtrack returned NaN (continental crust or any
# masked region along the profile) so the filled polygon below can
# close cleanly — without this the line breaks wherever it crosses
# unmodelled crust.
profile_clean = profile_track.dropna(subset=["depth"]).reset_index(drop=True)
print(f"Profile samples kept: {len(profile_clean)} / {len(profile_track)} "
      f"(dropped {len(profile_track) - len(profile_clean)} NaN points)")

fig = pygmt.Figure()
# Bump axis-label and tick-label fonts well above the suite default —
# at 36 cm wide they look tiny otherwise.
with pygmt.config(FONT_LABEL="28p", FONT_ANNOT_PRIMARY="22p",
                  FONT_TITLE="24p"):
    fig.basemap(
        region=[0, len(profile_clean), profile_clean["depth"].min()*1.1, -2000],
        projection="X36c/14c",
        frame=["xaf+lDistance (samples)", "yaf+lPaleo-depth (m)",
               f"WSrt+tNYC—Lisbon paleo-basement depth at {T} Ma (sediment-free)"])
fig.plot(x=np.arange(len(profile_clean)), y=profile_clean["depth"],
         fill="lightblue", close="+yB", pen="0.5p,black")
fig.show(width=1100)
display(HTML('<div style="height:1cm"></div>'))

## Extend this

- **Add sediment cover.** Layer a paleo-sediment-thickness grid on top of the basement to get true paleobathymetry.
- **Different age.** Re-render at 100 Ma (mid-Atlantic opening) or 20 Ma (fully opened) for comparison.
- **Switch cooling model.** Replace GDH with the plate-cooling model of Parsons & Sclater (1977) to see how the asymptotic depth limit changes the deep-basin signal.

## References

- Mather, B.R., Müller, R.D., Zahirovic, S., Cannon, J., Chin, M., Ilano, L., Wright, N.M., Alfonso, C., Williams, S., Tetley, M. & Merdith, A. (2024). Deep time spatio-temporal data analysis using GPlately. *Geosci. Data J.* 11, 3-10. https://doi.org/10.1002/gdj3.185
- Tian, D., Uieda, L., Leong, W.J., Fröhlich, Y., Schlitzer, W., Grund, M., Jones, M., Toney, L., Yao, J., Magen, Y., Materna, K., Belem, A., Newton, T., Anant, A., Ziebarth, M., Quinn, J., Wessel, P. (2024). PyGMT: A Python interface for the Generic Mapping Tools. *Zenodo*. https://doi.org/10.5281/zenodo.13679085
- Wessel, P., Luis, J.F., Uieda, L., Scharroo, R., Wobbe, F., Smith, W.H.F. & Tian, D. (2019). The Generic Mapping Tools version 6. *Geochem. Geophys. Geosys.* 20, 5556-5564. https://doi.org/10.1029/2019GC008515
- Chin, M., Mather, B.R. & Müller, R.D. (2024). Plate Model Manager: A Python package for downloading and managing published plate-reconstruction models. *Zenodo*. https://github.com/michaelchin/plate-model-manager
- Zahirovic, S., Eleish, A., Doss, S., Pall, J., Cannon, J., Pistone, M., Tetley, M.G., Young, A. & Müller, R.D. (2022). Subduction and carbonate platform interactions. *Geoscience Data Journal* 9, 371-383. https://doi.org/10.1002/gdj3.146
- Stein, C.A. & Stein, S. (1992). A model for the global variation in oceanic depth and heat flow with lithospheric age. *Nature* 359, 123-129. https://doi.org/10.1038/359123a0